In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

## 1- Loading Data

In [2]:
# loading data
top_impressions = pd.read_csv('top_impressions.csv', encoding='utf-8')
top_engagements = pd.read_csv('top_engagements.csv', encoding='utf-8')
classified_posts = pd.read_csv('classified_posts.csv', encoding='utf-8')

In [3]:
top_impressions.head(3)

,id,post_publish_date,day_name,impressions
0,7426169034179547136,2026-02-08,Sunday,24049
1,7434855171056144384,2026-03-04,Wednesday,18207
2,7435367669677101056,2026-03-05,Thursday,7270


In [4]:
top_engagements.head(3)

,id,post_publish_date,day_name,engagements
0,7426169034179547136,2026-02-08,Sunday,340
1,7434855171056144384,2026-03-04,Wednesday,223
2,7435367669677101056,2026-03-05,Thursday,133


In [5]:
classified_posts.head(3)

,id,attachments,post_len,category,style
0,7426169034179547136,N,503,Tech & Educational Content,Professional
1,7434855171056144384,N,462,Career Advice & Mindset,Funny
2,7435367669677101056,Img,2528,Career Advice & Mindset,Funny


In [6]:
# checking dataframes shapes
print(f'Top Impressions DF Shape: {top_impressions.shape}')
print(f'Top Engagements DF Shape: {top_engagements.shape}')
print(f'Classified Posts DF Shape: {classified_posts.shape}')

Top Impressions DF Shape: (50, 4)
Top Engagements DF Shape: (50, 4)
Classified Posts DF Shape: (63, 5)


In [7]:
# merging top_impressions df with classified_posts df
top_impressions_classified = pd.merge(top_impressions, classified_posts, how='left', left_on='id', right_on='id')
# merging top_engagements df with classified_posts df
top_engagements_classified = pd.merge(top_engagements, classified_posts, how='left', left_on='id', right_on='id')

## 2- Assessing Data

In [8]:
# checking for null values
print('Impressions Null Values: ', top_impressions_classified.isnull().sum().sum())
print('Engagements Null Values: ', top_engagements_classified.isnull().sum().sum())

Impressions Null Values:  0
Engagements Null Values:  0


In [9]:
# checking for duplicated rows
print('Impressions Duplicated Rows: ', top_impressions_classified.duplicated().sum())
print('Engagements Duplicated Rows: ', top_engagements_classified.duplicated().sum())

Impressions Duplicated Rows:  0
Engagements Duplicated Rows:  0


## 3- Data Preprocessing

In [10]:
# dropping id, post_publish_date columns
top_impressions_classified.drop(['id', 'post_publish_date'], axis=1, inplace=True)
top_engagements_classified.drop(['id', 'post_publish_date'], axis=1, inplace=True)

In [11]:
top_impressions_classified.head(3)

,day_name,impressions,attachments,post_len,category,style
0,Sunday,24049,N,503,Tech & Educational Content,Professional
1,Wednesday,18207,N,462,Career Advice & Mindset,Funny
2,Thursday,7270,Img,2528,Career Advice & Mindset,Funny


In [12]:
top_engagements_classified.head(3)

,day_name,engagements,attachments,post_len,category,style
0,Sunday,340,N,503,Tech & Educational Content,Professional
1,Wednesday,223,N,462,Career Advice & Mindset,Funny
2,Thursday,133,Img,2528,Career Advice & Mindset,Funny


### 3.1- Reducing Attachments Dimensionality

In [13]:
# replacing 'share' attribute with 'N'
top_impressions_classified.attachments = top_impressions_classified.attachments.replace('Share', 'N')
top_engagements_classified.attachments = top_engagements_classified.attachments.replace('Share', 'N')

In [14]:
# displaying attachments unique values in top impressions dataframe
top_impressions_classified.groupby('attachments').style.count()

attachments
Img     17
Imgs     8
N       14
Vid     11
Name: style, dtype: int64

In [15]:
# displaying attachments unique values in top engagements dataframe
top_engagements_classified.groupby('attachments').style.count()

attachments
Img     21
Imgs     8
N       13
Vid      8
Name: style, dtype: int64

### 3.2- Categorical Variables One-Hot Encoding

In [16]:
pd.get_dummies(top_impressions_classified).head(3)

,impressions,post_len,day_name_Friday,day_name_Monday,day_name_Saturday,day_name_Sunday,day_name_Thursday,day_name_Tuesday,day_name_Wednesday,attachments_Img,attachments_Imgs,attachments_N,attachments_Vid,category_Career Advice & Mindset,category_Career Updates & Achievements,category_Tech & Educational Content,style_Funny,style_Personal,style_Professional
0,24049,503,False,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,True
1,18207,462,False,False,False,False,False,False,True,False,False,True,False,True,False,False,True,False,False
2,7270,2528,False,False,False,False,True,False,False,True,False,False,False,True,False,False,True,False,False


In [17]:
# dropping a specific column from each category
top_impressions_dum = pd.get_dummies(top_impressions_classified).drop(['day_name_Monday', 'attachments_N', 'category_Career Advice & Mindset', 'style_Personal'], axis=1)
top_engagements_dum = pd.get_dummies(top_engagements_classified).drop(['day_name_Monday', 'attachments_N', 'category_Career Advice & Mindset', 'style_Personal'], axis=1)

In [18]:
# Replacing True/False Values to 1/0
top_impressions_dum = top_impressions_dum.replace({True:1, False:0}).infer_objects(copy=False)
top_engagements_dum = top_engagements_dum.replace({True:1, False:0}).infer_objects(copy=False)

C:\Users\TOSHIBA\AppData\Local\Temp\ipykernel_10984\3873278988.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  top_impressions_dum = top_impressions_dum.replace({True:1, False:0}).infer_objects(copy=False)
C:\Users\TOSHIBA\AppData\Local\Temp\ipykernel_10984\3873278988.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  top_engagements_dum = top_engagements_dum.replace({True:1, False:0}).infer_objects(copy=False)


### 3.3- Log-Transforming Targets

In [19]:
top_impressions_dum.impressions = np.log(top_impressions_dum.impressions)
top_engagements_dum.engagements = np.log(top_engagements_dum.engagements)

## 4- Regularized Linear Regression (Elastic Net)

In [20]:
test_ratios = np.arange(0.0001, 0.1, 0.0001)

In [21]:
elastic_model = ElasticNetCV(l1_ratio=test_ratios, cv=5, random_state=42)

### 4.1- Impressions

In [22]:
# extracting dependent & independent variables
x_imp = top_impressions_dum.drop('impressions', axis=1)
y_imp = top_impressions_dum.impressions

In [23]:
# standardizing inputs
scaler = StandardScaler()
x_imp_scaled = scaler.fit_transform(x_imp)

In [24]:
elastic_model.fit(x_imp_scaled, y_imp)

,l1_ratio,"array([0.0001...0998, 0.0999])"
,eps,0.001
,n_alphas,'deprecated'
,alphas,'warn'
,fit_intercept,True
,precompute,'auto'
,max_iter,1000
,tol,0.0001
,cv,5
,copy_X,True
,verbose,0


In [25]:
# Printint the best parameters found by the cross-validation
print("Best Alpha:", elastic_model.alpha_)
print("Best L1 Ratio:", elastic_model.l1_ratio_)

Best Alpha: 1.6840998816688386
Best L1 Ratio: 0.0825


In [26]:
# Printint the best parameters found by the cross-validation
print("Best Alpha:", elastic_model.alpha_)
print("Best L1 Ratio:", elastic_model.l1_ratio_)

Best Alpha: 1.6840998816688386
Best L1 Ratio: 0.0825


In [27]:
print('Intercept: ', elastic_model.intercept_)
for i in range(len(elastic_model.coef_)):
    print(f'{x_imp.columns[i]}: {elastic_model.coef_[i]}')

Intercept:  6.778229412211708
post_len: 0.0
day_name_Friday: -0.0
day_name_Saturday: -0.0
day_name_Sunday: 0.047765063743184434
day_name_Thursday: 0.0
day_name_Tuesday: -0.0
day_name_Wednesday: 0.0
attachments_Img: -0.0
attachments_Imgs: -0.0
attachments_Vid: 0.0
category_Career Updates & Achievements: 0.0
category_Tech & Educational Content: -0.0
style_Funny: -0.0
style_Professional: 0.01639748712380619


### 4.2- Engagements Model

In [28]:
# extracting dependent & independent variables
x_eng = top_engagements_dum.drop('engagements', axis=1)
y_eng = top_engagements_dum.engagements

In [29]:
# standardizing inputs
scaler = StandardScaler()
x_eng_scaled = scaler.fit_transform(x_eng)

In [30]:
elastic_model.fit(x_eng_scaled, y_eng)

,l1_ratio,"array([0.0001...0998, 0.0999])"
,eps,0.001
,n_alphas,'deprecated'
,alphas,'warn'
,fit_intercept,True
,precompute,'auto'
,max_iter,1000
,tol,0.0001
,cv,5
,copy_X,True
,verbose,0


In [31]:
# Printint the best parameters found by the cross-validation
print("Best Alpha:", elastic_model.alpha_)
print("Best L1 Ratio:", elastic_model.l1_ratio_)

Best Alpha: 10.24344223028715
Best L1 Ratio: 0.0078000000000000005


In [32]:
print('Intercept: ', elastic_model.intercept_)
for i in range(len(elastic_model.coef_)):
    print(f'{x_eng.columns[i]}: {elastic_model.coef_[i]}')

Intercept:  2.7086484799987773
post_len: 0.0
day_name_Friday: 0.0
day_name_Saturday: -0.0
day_name_Sunday: 0.009349633175448811
day_name_Thursday: 0.0
day_name_Tuesday: -0.0
day_name_Wednesday: -0.0
attachments_Img: -0.00847529512107753
attachments_Imgs: 0.0
attachments_Vid: 0.0
category_Career Updates & Achievements: 0.0
category_Tech & Educational Content: -0.0076578537011718325
style_Funny: -0.0
style_Professional: 0.0


## 5- Interpretation

### 5.1- Impressions

1. Day of Week: Posting on Sunday increases impressions by 4.77% compared to other days of the week.
2. Post Style: Professional post style increases impressions by 1.64% compared to other styles.

### 5.2- Engagements

1. Day of Week: Posting on Sunday increases engagements by 0.93% compared to other days of the week.
2. Attachments: Attaching a single Image increases engagements by 0.85% compared to videos, multiple images or no attachments.
3. Post Category: Educational Content increases engagements by 0.77% compared to other categories.